# SO-101: Forward and Inverse Kinematics — **real Menagerie meshes**

This notebook renders the **original SO-101 CAD/STL mesh geometry** using MuJoCo WebAssembly + Three.js, while preserving the same V4 Python widget communication API across Binder, Colab, and VS Code.

**Learning objectives:** explore joint limits and transforms; compute 3D FK, a numerical Jacobian, and position IK; compare predicted and simulated motion.

The mesh source is [Google DeepMind MuJoCo Menagerie (`robotstudio_so101`)](https://github.com/google-deepmind/mujoco_menagerie/tree/main/robotstudio_so101). Model and meshes are downloaded over HTTPS at runtime. The actual robot visual meshes and MuJoCo collision geometries are preserved; a non-colliding floor and optional IK target are added for teaching. The browser must be online and able to fetch raw GitHub assets. The 3D viewer does not yet render textures, but Menagerie's SO-101 uses material colors for its CAD meshes.

**Important:** Before sharing with students, update `REPO_URL` in the setup cell. The new JavaScript frontend must be built and committed to `dist/viewer.js` once by the instructor. Widget communications are nonblocking: do **not** use `await viewer.get_state()` in Colab.

## 1. Setup — same notebook for Binder, Colab, and local Jupyter

**Instructor:** change `REPO_URL` once to the repository containing `mujoco_widget.py`, `so101_kinematics.py`, `dist/viewer.js`, `requirements.txt`, and this notebook. This follows the working V4 setup. New Colab sessions clone the repo automatically; Binder and local Jupyter import the project directly.

In [ ]:
import os, sys, subprocess
from pathlib import Path
from urllib.parse import urlparse

# INSTRUCTOR: Change this one URL before sharing the notebook.
REPO_URL = "https://github.com/szafird/mujoco-example.git"
parsed = urlparse(REPO_URL)
parts = parsed.path.strip('/').removesuffix('.git').split('/')
if parsed.netloc != 'github.com' or len(parts) != 2 or 'YOUR_' in REPO_URL:
    raise ValueError("Instructor: set REPO_URL to your public GitHub repository first")
owner, repo = parts
ASSET_BASE = f"https://{owner}.github.io/{repo}/mujoco/"

if 'google.colab' in sys.modules:
    repo_dir = Path('/content/so101_course_repo')
    if not repo_dir.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_dir)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo_dir / 'requirements.txt')], check=True)
    from google.colab import output
    output.enable_custom_widget_manager()
    sys.path.insert(0, str(repo_dir))
else:
    repo_dir = Path.cwd()
    # Allow opening notebook from a notebooks/ subfolder.
    if not (repo_dir / 'mujoco_widget.py').exists() and (repo_dir.parent / 'mujoco_widget.py').exists():
        repo_dir = repo_dir.parent
    sys.path.insert(0, str(repo_dir))

from mujoco_widget import MuJoCoViewer
from so101_kinematics import SO101Kinematics, load_menagerie_xml, make_mesh_scene_xml, MENAGERIE_SO101_ASSETS, ALL_JOINTS
import numpy as np
import ipywidgets as widgets
from IPython.display import display

print('Python project:', repo_dir)
print('Hosted MuJoCo assets:', ASSET_BASE)

## 2. Load authentic SO-101 XML and mesh assets

The notebook fetches Menagerie's original MJCF and uses its exact joints, bodies, actuator definitions, mesh geoms, and material colors. The browser downloads the referenced STLs from Menagerie at initialization. The first load may take longer than the primitive-only V4 scene.

In [ ]:
cache = repo_dir / 'models' / 'so101.xml'
if cache.exists():
    original_xml = cache.read_text(encoding='utf-8')
else:
    print('Fetching robotstudio_so101/so101.xml from MuJoCo Menagerie...')
    original_xml = load_menagerie_xml()

robot = SO101Kinematics(original_xml)
print('Joint names:', robot.joint_names)
print('Joint limits (radians):')
for name, (lo, hi) in zip(robot.joint_names, robot.limits):
    print(f'  {name:<15} [{lo:+.3f}, {hi:+.3f}]')

## 3. View the actual SO-101 CAD meshes

Wait for **Ready** (the toolbar will first show a mesh-download/compilation status). Orbit with drag and zoom with scroll. If loading fails, read the status text and open the browser console. The notebook and JavaScript frontend use the *same* V4 nonblocking command API in Binder and Colab.

In [ ]:
scene_xml = make_mesh_scene_xml(original_xml)
viewer = MuJoCoViewer(
    scene_xml, asset_base=ASSET_BASE,
    mesh_source_url=MENAGERIE_SO101_ASSETS,
    camera_distance=0.85, camera_target=[0.0, 0.0, 0.16]
)
display(viewer)

### Joint-angle sliders

Each slider commands a Menagerie SO-101 **position actuator**. The robot should now visibly move with its real STL/CAD geometry. The site `gripperframe` is used for numerical FK, though the mesh viewport currently does not draw the coordinate frame itself.

In [ ]:
sliders = [widgets.FloatSlider(value=0.0, min=float(lim[0]), max=float(lim[1]),
            step=0.02, description=name, readout_format='.2f',
            style={'description_width':'125px'}, layout=widgets.Layout(width='440px'))
           for name, lim in zip(robot.joint_names, robot.limits)]

def send_joint_targets(change=None):
    for i, slider in enumerate(sliders):
        viewer.set_control(i, float(slider.value))

for slider in sliders:
    slider.observe(send_joint_targets, names='value')
display(widgets.VBox(sliders))
send_joint_targets()

## 4. Forward kinematics: joint angles → end-effector pose

The arm has five joints controlling the pose and a sixth opening the gripper. We compute the gripper-frame world pose from the original Menagerie **body transforms** and **joint rotations**:

$$T_{WE}(q) = T_{WB_1}(q_1)\,T_{B_1B_2}(q_2)\cdots T_{B_nE}(q_n).$$

Each joint is a rotation about its locally defined axis. The `SO101Kinematics.fk()` implementation traverses the parsed MJCF hierarchy, composes homogeneous transforms, and returns the 4×4 end-effector pose matrix.

In [ ]:
q = np.array([slider.value for slider in sliders], dtype=float)
T = robot.fk(q)
np.set_printoptions(precision=4, suppress=True)
print('Commanded joint angles (radians):', q)
print('FK: world T gripperframe =\n', T)
print('Predicted gripper position (m):', T[:3, 3])

**Experiment:** Modify `q` below and compare the predicted gripper position. You can also command the simulator using the same angles. The physical arm approaches commanded positions over time because it uses the model's servos and dynamics.

In [ ]:
q_test = np.array([0.55, -0.45, 0.65, 0.25, 0.1, 0.35])
print('Predicted end-effector xyz:', robot.xyz(q_test))
for joint, angle in zip(sliders, q_test):
    joint.value = float(angle)
# The viewer will move as the position actuators track these targets.

### Read actual simulated joint positions

The simulator reports actual joint positions in `qpos`, which may lag behind commanded angles. Request the snapshot in one cell, then inspect it in the **next cell**. Both Binder and Colab use this exact nonblocking pattern.

In [ ]:
viewer.request_sim_state()  # nonblocking; run the next cell after this finishes

In [ ]:
state = viewer.latest_sim_state
if state is None:
    print('No snapshot yet: viewer must be Ready; rerun the previous request cell.')
else:
    q_actual = np.array(state['qpos'][:6])
    print('Actual qpos:', q_actual)
    print('Actual gripper xyz (from FK on measured q):', robot.xyz(q_actual))
    print('Desired gripper xyz (from sliders):', robot.xyz([s.value for s in sliders]))

## 5. Jacobian: how does moving a joint move the gripper?

For position-only tasks, the Jacobian is $J_p(q)\in\mathbb{R}^{3\times 5}$:

$$\dot p = J_p(q)\,\dot q,\qquad
J_{p,ij} \approx \frac{p_i(q+\epsilon e_j)-p_i(q-\epsilon e_j)}{2\epsilon}.$$

The gripper-opening joint is excluded from the position IK. This finite-difference Jacobian is simple to inspect; an advanced exercise is to derive the geometric Jacobian analytically from the joint axes.

In [ ]:
q = np.array([s.value for s in sliders], dtype=float)
J = robot.jacobian(q)
print('Jacobian shape:', J.shape)
print(J)
print('Singular values:', np.linalg.svd(J, compute_uv=False))

## 6. Inverse kinematics: target xyz → joint angles

We solve **position-only IK** (3D xyz, not full 6D pose) using **damped least squares**:

$$\Delta q = J^T(JJ^T + \lambda^2 I)^{-1}\,(p^* - p(q)).$$

A reachable target is generated from a known configuration using FK. This avoids accidentally choosing an unreachable point. The solver clips each joint to the Menagerie joint limits and leaves the gripper opening unchanged.

In [ ]:
# A reachable target (generated by FK) for the class exercise.
q_goal = np.array([0.60, -0.65, 0.82, 0.38, -0.18, 0.25])
target_xyz = robot.xyz(q_goal)
print('Target xyz (m):', target_xyz)

seed = np.zeros(6)
solution = robot.ik_position(target_xyz, seed=seed)
print('Converged:', solution.reached)
print('Iterations:', solution.iterations)
print('Final position error (m):', solution.error)
print('IK joint solution:', solution.q)
print('FK(solution):', robot.xyz(solution.q))
assert solution.reached, 'IK did not converge; adjust damping/seed or target'

### Visualize the IK target and move the rendered robot

This creates a second viewer using the original CAD meshes, adding a noncolliding purple target sphere at the requested gripper position.

In [ ]:
ik_scene = make_mesh_scene_xml(original_xml, target=target_xyz)
ik_viewer = MuJoCoViewer(
    ik_scene, asset_base=ASSET_BASE,
    mesh_source_url=MENAGERIE_SO101_ASSETS,
    camera_distance=0.85, camera_target=[0.0, 0.0, 0.16]
)
display(ik_viewer)

In [ ]:
for i, position in enumerate(solution.q):
    ik_viewer.set_control(i, float(position))
ik_viewer.play()
print('Commanded IK solution; watch the arm move toward the purple target.')

### Verify the achieved error from simulation state

Run the request, then inspect in a subsequent cell. The gripper-frame position should approach the target; model dynamics, joint friction, servo saturation, and solver approximations can cause nonzero tracking error.

In [ ]:
ik_viewer.request_sim_state()

In [ ]:
state = ik_viewer.latest_sim_state
if state:
    measured_xyz = robot.xyz(np.asarray(state['qpos'][:6]))
    print('Target:', target_xyz)
    print('Actual:', measured_xyz)
    print('Position error (m):', np.linalg.norm(target_xyz - measured_xyz))
else:
    print('No state yet. Wait for Ready and request it again.')

## 7. Exercises

1. **Forward kinematics:** Use the sliders to find a configuration that moves the gripper roughly 5 cm along the world x-axis. Record the start/end joint vectors, FK poses, and measured change.
2. **Jacobian validation:** Pick a small joint displacement $\Delta q$ and compare $J_p(q)\Delta q$ with $p(q+\Delta q)-p(q)$. How does the approximation change with step size?
3. **Inverse kinematics:** Change `q_goal` (within the joint limits) to create three reachable target positions. Solve each with different initial guesses. Compare solutions, errors, and iteration counts.
4. **Singularities:** Find a configuration where the smallest singular value of the position Jacobian is small. Explain why IK becomes more difficult.
5. **Extension:** Upgrade position-only IK to full pose (position and orientation), including an orientation error and a 6×5 Jacobian. What task constraints can a five-joint arm satisfy exactly?

**Instructor notes:** This model has five arm joints plus a gripper joint. With five independent arm joints, arbitrary 6D end-effector poses are generally not all reachable. The gripper joint changes the fingers but is not an independent pose degree of freedom for the gripper reference frame. No collision avoidance is included in the IK solver. When using the browser viewer, avoid blocking cells (`time.sleep`, busy-wait loops, or `await` on a widget reply) as Colab may defer custom-comm callbacks until execution finishes.